In [ ]:
# Install MetaTrader5 package
# !pip install MetaTrader5==5.0.43

In [ ]:
# Import the libraries
import MetaTrader5 as mt5
import numpy as np
import pandas as pd
import time
import winsound
import warnings
warnings.filterwarnings("ignore")
from time import strftime

In [ ]:
# Initialize variables
symbol = 'XAUUSD_i'
timeframe = mt5.TIMEFRAME_M30
number_of_data = 1000
lot = 0.01
deviation = 8

In [ ]:
# Bound to MT5
mt5.initialize()

In [ ]:
''' Main Loop '''
# .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

while True:

    ''' Get data '''
    # .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

    # Extract n Ticks before now
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 1, number_of_data)

    # Transform Tuple into a DataFrame
    df = pd.DataFrame(rates)

    # Convert the number format of the date into the date format
    df['time'] = pd.to_datetime(df['time'], unit='s')

    # Set index column
    df = df.set_index('time')

    ''' Prediction '''
    # .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

    # Last Prices
    Last_Price = df.iloc[-1]['close']
    Second_Last_Price = df.iloc[-2]['close']

    # EMA(20 - 50)
    df['EWMA-20'] = df['close'].ewm(span=20).mean()
    df['EWMA-50'] = df['close'].ewm(span=50).mean()

    ''' Signal '''
    # .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

    # Signal Buy
    con_1_buy = df['EWMA-20'].iloc[-1] > df['EWMA-20'].iloc[-2]
    con_2_buy = df['EWMA-50'].iloc[-1] > df['EWMA-50'].iloc[-2]
    con_3_buy = df['EWMA-20'].iloc[-1] > df['EWMA-50'].iloc[-1]
    con_4_buy = Last_Price > Second_Last_Price
    con_5_buy = Last_Price > df['EWMA-20'].iloc[-1]

    # Signal Sell
    con_1_sell = df['EWMA-20'].iloc[-1] < df['EWMA-20'].iloc[-2]
    con_2_sell = df['EWMA-50'].iloc[-1] < df['EWMA-50'].iloc[-2]
    con_3_sell = df['EWMA-20'].iloc[-1] < df['EWMA-50'].iloc[-1]
    con_4_sell = Last_Price < Second_Last_Price
    con_5_sell = Last_Price < df['EWMA-20'].iloc[-1]

    # Create signal
    buy = False
    sell = False

    if con_1_buy and con_2_buy and con_3_buy and con_4_buy and con_5_buy:
        buy = True
    elif con_1_sell and con_2_sell and con_3_sell and con_4_sell and con_5_sell:
        sell = True

    ''' SL and TP '''
    # .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

    # Compute the Stop Loss(SL) and Take Profit(TP)
    risk = 1
    reward = 2

    # Extract the leverage
    leverage = mt5.account_info().leverage

    # Compute the price
    price = mt5.symbol_info(symbol).ask

    # Extract the number of decimals
    nb_decimal = str(price)[::-1].find('.')

    # Compute the variations in percentage
    var_down = risk / leverage
    var_up = reward / leverage

    # Find the TP and SL threshold in absolute price
    if buy:
        price = mt5.symbol_info(symbol).ask

        # Compute the variations in absolute price
        price_var_down = var_down * price
        price_var_up = var_up * price

        tp = np.round(price + price_var_up, nb_decimal)
        sl = np.round(price - price_var_down, nb_decimal)

    elif sell:
        price = mt5.symbol_info(symbol).bid

        # Compute the variations in absolute price
        price_var_down = var_down * price
        price_var_up = var_up * price

        tp = np.round(price - price_var_up, nb_decimal)
        sl = np.round(price + price_var_down, nb_decimal)

    ''' Send Order '''
    # .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

    # get open positions on current symbol
    position = mt5.positions_get(symbol=symbol)

    # Sending order
    if buy and len(position)==0:

        request = {
        'action': mt5.TRADE_ACTION_DEAL,
        'symbol': symbol,
        'volume': lot,
        'type': mt5.ORDER_TYPE_BUY,
        'price': mt5.symbol_info_tick(symbol).ask,
        'deviation': deviation,
        'tp': tp,
        'sl': sl, 
        'type_filling': mt5.ORDER_FILLING_IOC,
        'type_time': mt5.ORDER_TIME_GTC}

        # Send order
        mt5.order_send(request)

        # Print Specifications
        print(f'Symbol: {symbol} | Buy  | {strftime("%Y-%m-%d %H:%M:%S")}')  

        # Initialize the Last Price for trailing SL
        last_price = mt5.symbol_info_tick(symbol).bid
        
        # Make beep sound on Windows
        for i in range(3):
            winsound.Beep(frequency=2000, duration=500)
            time.sleep(0.5)
        print(20*'-^-')

    # Sending order   
    if sell and len(position)==0:

        request = {
        'action': mt5.TRADE_ACTION_DEAL,
        'symbol': symbol,
        'volume': lot,
        'type': mt5.ORDER_TYPE_SELL,
        'price': mt5.symbol_info_tick(symbol).bid,
        'deviation': deviation,
        'tp': tp,
        'sl': sl, 
        'type_filling': mt5.ORDER_FILLING_IOC,
        'type_time': mt5.ORDER_TIME_GTC}

        # Send order
        mt5.order_send(request)

        # Print Specifications
        print(f'Symbol: {symbol} | Sell | {strftime("%Y-%m-%d %H:%M:%S")}')

        # Initialize the Last Price for trailing SL
        last_price = mt5.symbol_info_tick(symbol).ask
        
        # Make beep sound on Windows
        for i in range(3):
            winsound.Beep(frequency=2000, duration=500)
            time.sleep(0.5)
        print(20*'-^-')

    ''' Trailing SL '''
    # .^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.^.

    # Trailing SL(Buy)
    if len(position)!=0 and position[0][5]==0:

        # Current price
        current_price = mt5.symbol_info(symbol).bid
        
        # Difference between current price and last price
        # we use 'try', and 'except' if we set the order manually(last_price)
        try:
            if current_price <= last_price:
                difference = 0
            elif current_price > last_price:
                difference = current_price - last_price
                last_price = current_price
        except:
            # Initialize the Last Price for trailing SL
            position = mt5.positions_get(symbol=symbol)
            last_price = position[0][10] # price_open
            if current_price <= last_price:
                difference = 0
            elif current_price > last_price:
                difference = current_price - last_price
                last_price = current_price

        # Change the sl
        request = {
        'action': mt5.TRADE_ACTION_SLTP,
        'symbol': position[0][16],
        'position': position[0][0],
        'volume': position[0][9],
        'type': mt5.ORDER_TYPE_BUY,
        'price': position[0][10],
        'tp': position[0][12],
        'sl': position[0][11] + difference,
        'type_filling': mt5.ORDER_FILLING_IOC,
        'type_time': mt5.ORDER_TIME_GTC,
        }

        mt5.order_send(request)                

    # Trailing SL(Sell)
    if len(position)!=0 and position[0][5]==1:

        # Current price
        current_price = mt5.symbol_info(symbol).ask
        
        # Difference between current price and last price
        # we use 'try', and 'except' if we set the order manually(last_price)
        try:
            if current_price >= last_price:
                difference = 0
            elif current_price < last_price:
                difference = last_price - current_price
                last_price = current_price
        except:
            # Initialize the Last Price for trailing SL
            position = mt5.positions_get(symbol=symbol)
            last_price = position[0][10] # price_open
            if current_price >= last_price:
                difference = 0
            elif current_price < last_price:
                difference = last_price - current_price
                last_price = current_price

        # Change the sl
        request = {
        'action': mt5.TRADE_ACTION_SLTP,
        'symbol': position[0][16],
        'position': position[0][0],
        'volume': position[0][9],
        'type': mt5.ORDER_TYPE_SELL,
        'price': position[0][10],
        'tp': position[0][12],
        'sl': position[0][11] - difference,
        'type_filling': mt5.ORDER_FILLING_IOC,
        'type_time': mt5.ORDER_TIME_GTC,
        }

        mt5.order_send(request) 
